# Preventive Health Assistant (RAG Prototype)

## 1. Motivation

In the main analytics project, several factors were identified as strong drivers of healthcare costs, particularly:

* smoking
* high Body Mass Index (BMI)
* age

However, predictive models explain which variables influence costs, but they do not fully explain **why these factors affect health outcomes**.

To complement the analysis, this notebook explores a simple **Retrieval-Augmented Generation** (RAG) approach that allows us to build a small assistant capable of answering questions about:

* **prevention strategies** based on insights from the main project
* health knowledge about **lifestyle risks**

The goal is to demonstrate **how analytics and AI can work together** to support prevention-oriented health insights.

This prototype is a **proof-of-concept** created for educational purposes and *should not be interpreted as medical advice*.


## 2. Knowledge Base

To build the assistant, we create a small knowledge base containing **short documents related to lifestyle risk factors** and **preventive healthcare**.

These documents summarize well-known relationships between behaviors such as:

* smoking and chronic disease  
* obesity and metabolic risk  
* preventive healthcare strategies  
* lifestyle interventions and long-term health outcomes  

When a question is asked, the assistant retrieves the most relevant information from this knowledge base.

In [1]:
documents = [
"Smoking significantly increases the risk of cardiovascular disease, respiratory illness, and multiple types of cancer. Long-term smokers also tend to require more medical care over time, increasing healthcare utilization and costs.",
"High Body Mass Index (BMI) is associated with metabolic conditions such as type 2 diabetes, hypertension, and cardiovascular disease. These chronic conditions often lead to higher long-term healthcare spending.",
"Preventive healthcare strategies include smoking cessation programs, regular physical activity, balanced nutrition, and early disease screening. These interventions can reduce long-term health risks.",
"Lifestyle interventions such as improved diet, regular exercise, and weight management can significantly reduce the risk of chronic diseases and improve long-term health outcomes.",
"Smoking cessation programs are among the most effective public health interventions. Individuals who stop smoking reduce their risk of cardiovascular disease, lung disease, and cancer over time.",
"Early prevention strategies can reduce healthcare costs by lowering the incidence of chronic disease. Population health programs often focus on behavioral risk factors such as smoking, obesity, and inactivity."
]

## 3. Creating Document Embeddings

To enable semantic search, the documents are converted into **vector embeddings** using a pre-trained SentenceTransformer model.

These embeddings allow the system to measure the semantic similarity between a user question and the stored documents.

In [2]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = model.encode(documents)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

## 4. Retrieving Relevant Information

When a user asks a question, the system converts the question into an embedding and compares it with the document embeddings.

The assistant then retrieves the **most relevant passages** from the knowledge base based on semantic similarity.

In [3]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def retrieve(query, top_k=2):
    query_embedding = model.encode([query])
    similarities = cosine_similarity(query_embedding, doc_embeddings)[0]
    top_indices = similarities.argsort()[-top_k:][::-1]
    results = [(documents[i], similarities[i]) for i in top_indices]

    return results

## 5. Generating a Response

After retrieving the most relevant documents, the assistant returns the retrieved information as context to answer the question.

In a full RAG system, a large language model would generate a detailed answer.  
In this simplified prototype, we display the most relevant passages retrieved from the knowledge base.

In [4]:
def answer_question(question):
    retrieved_docs = retrieve(question)

    print("Question:", question)
    print("\nRelevant information:\n")

    for doc in retrieved_docs:
        print("-", doc)

The assistant can be used to answer questions such as:

* Why does smoking increase healthcare costs?  
* What lifestyle factors contribute most to chronic disease?  
* What prevention strategies reduce long-term healthcare expenses?  
* How can preventive health programs improve population health outcomes?

## 6. Try the Assistant

You can interact with the assistant by typing your own health-related questions.

This allows you to explore how the RAG system retrieves relevant information from the knowledge base.

In [5]:
answer_question("Why does smoking increase healthcare costs?")

Question: Why does smoking increase healthcare costs?

Relevant information:

- ('Smoking significantly increases the risk of cardiovascular disease, respiratory illness, and multiple types of cancer. Long-term smokers also tend to require more medical care over time, increasing healthcare utilization and costs.', np.float32(0.67258257))
- ('Early prevention strategies can reduce healthcare costs by lowering the incidence of chronic disease. Population health programs often focus on behavioral risk factors such as smoking, obesity, and inactivity.', np.float32(0.5828413))
